# LeetCode 946 — Validate Stack Sequences bằng Python

Link đề: https://leetcode.com/problems/validate-stack-sequences/ · Độ khó: **Medium** · Chủ đề: Array, Stack, Simulation

Skeleton LeetCode:

```python
class Solution:
    def validateStackSequences(self, pushed: List[int], popped: List[int]) -> bool:
        ...
```

Ghi chú: đáp án là `bool`. Điểm cốt lõi: các giá trị **phân biệt** nên chiến lược **tham lam** là đúng — đẩy lần lượt các phần tử của `pushed`, và **ngay khi đỉnh ngăn xếp bằng phần tử cần pop kế tiếp thì pop**; cuối cùng ngăn xếp phải rỗng. Lý do: nếu đỉnh trùng phần tử cần pop thì pop là **lựa chọn duy nhất khả thi** (đẩy thêm sẽ chôn nó). Oracle: **tìm kiếm vét cạn mọi dãy thao tác** `push`/`pop` (nhánh cả hai lựa chọn), không giả định tham lam.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho dãy `pushed` (thứ tự đẩy) và `popped` (thứ tự lấy ra) của các giá trị phân biệt; hỏi có tồn tại dãy thao tác `push`/`pop` trên ngăn xếp ban đầu rỗng cho ra đúng hai dãy đó.
2. Nắm công cụ Python: `list` làm ngăn xếp, hai con trỏ `i` (đẩy) và `j` (lấy), tìm kiếm đệ quy với bộ nhớ, **mẫu hoán vị bị cấm** (pattern avoidance).
3. Đi từ **mô phỏng tham lam** → **dùng chính mảng `pushed` làm ngăn xếp (`O(1)` bộ nhớ)** → **kiểm tra mẫu `312` bị cấm** → **quy hoạch động / BFS trên trạng thái `(i, j)`**.
4. Hiểu mấu chốt: khi đỉnh ngăn xếp bằng `popped[j]`, **phải** pop ngay (mọi lựa chọn khác làm phần tử đó bị chôn dưới phần tử mới và không bao giờ ra đúng lúc), nên không cần quay lui.
5. Tránh lỗi kinh điển: pop chỉ **một lần** (dùng `if` thay `while`), quên kiểm tra ngăn xếp rỗng cuối cùng, đẩy hết rồi mới pop, so sánh sai chỉ số `popped`, và quên rằng giá trị phải phân biệt thì tham lam mới đúng.

## 2. Đề bài & ví dụ

Cho hai mảng số nguyên `pushed` và `popped`, mỗi mảng gồm các giá trị **phân biệt**.

Trả về `true` nếu đây có thể là kết quả của một dãy thao tác `push` và `pop` trên một ngăn xếp **ban đầu rỗng**; ngược lại `false`. (`pushed` là thứ tự các phần tử được đẩy vào; `popped` là thứ tự các phần tử được lấy ra.)

**Ví dụ 1:** `pushed = [1,2,3,4,5]`, `popped = [4,5,3,2,1]` → `true`: `push(1), push(2), push(3), push(4), pop → 4, push(5), pop → 5, pop → 3, pop → 2, pop → 1`.

**Ví dụ 2:** `pushed = [1,2,3,4,5]`, `popped = [4,3,5,1,2]` → `false` (`1` không thể được lấy ra trước `2`).

**Ràng buộc:** `1 <= pushed.length <= 1000`; `0 <= pushed[i] <= 1000`; các phần tử của `pushed` **phân biệt**; `popped.length == pushed.length`; `popped` là một **hoán vị** của `pushed`.

In [1]:
import itertools
import random
import time
from functools import lru_cache
from typing import List

examples = [(([1, 2, 3, 4, 5], [4, 5, 3, 2, 1]), True), (([1, 2, 3, 4, 5], [4, 3, 5, 1, 2]), False)]
for (a, b), want in examples:
    print(f"pushed = {a}, popped = {b} -> {want}")

pushed = [1, 2, 3, 4, 5], popped = [4, 5, 3, 2, 1] -> True
pushed = [1, 2, 3, 4, 5], popped = [4, 3, 5, 1, 2] -> False


## 3. Kiến thức nền

### 3.1 Ngăn xếp LIFO

`push(x)` đẩy lên đỉnh; `pop()` lấy phần tử **đỉnh** (vào sau, ra trước). Do đó phần tử bị lấy ra luôn là đỉnh tại thời điểm đó: khi nào `a` đẩy trước `b` và `b` còn trong ngăn xếp thì `a` không thể ra trước `b`.

In [2]:
stack = []
for op in ["push 1", "push 2", "pop", "push 3", "pop", "pop"]:
    if op.startswith("push"):
        stack.append(int(op.split()[1]))
        print(f"{op:7s} -> {stack}")
    else:
        print(f"{op:7s} -> lấy {stack.pop()}, còn {stack}")

push 1  -> [1]
push 2  -> [1, 2]
pop     -> lấy 2, còn [1]
push 3  -> [1, 3]
pop     -> lấy 3, còn [1]
pop     -> lấy 1, còn []


### 3.2 Hai con trỏ `i` và `j`

`i` là chỉ số phần tử kế tiếp của `pushed` cần đẩy; `j` là chỉ số phần tử kế tiếp của `popped` cần lấy ra. Trạng thái `(i, j)` cùng thứ tự thao tác trước đó xác định ngăn xếp: nội dung ngăn xếp = `pushed[:i]` **trừ** `popped[:j]` (giữ thứ tự đẩy).

### 3.3 Tại sao tham lam đúng (gợi ý)

Giá trị **phân biệt**. Giả sử đỉnh ngăn xếp bằng `popped[j]`. Nếu ta **không** pop mà đẩy thêm một phần tử `y`, thì `popped[j]` nằm **dưới** `y`: nó chỉ ra được sau khi `y` ra, nhưng `y` còn phải ra **sau** `popped[j]` (vì `popped[j]` cần ra tiếp theo) — mâu thuẫn. Nên pop ngay là lựa chọn **bắt buộc**.

### 3.4 Mẫu hoán vị bị cấm

Đánh số `pushed` là `0, 1, …, n−1`; dãy `q` là vị trí trong `pushed` của từng phần tử theo thứ tự `popped`. Các dãy lấy ra hợp lệ chính là các hoán vị **tránh mẫu `312`**: không tồn tại `a < b < c` (chỉ số trong `q`) với `q_b < q_c < q_a`. Số hoán vị hợp lệ độ dài `n` là số **Catalan** `C_n`.

In [3]:
def pushed_positions(pushed, popped):
    pos = {v: i for i, v in enumerate(pushed)}
    return [pos[v] for v in popped]


print(pushed_positions([1, 2, 3, 4, 5], [4, 5, 3, 2, 1]), "(hợp lệ)", pushed_positions([1, 2, 3, 4, 5], [4, 3, 5, 1, 2]), "(chứa 3,1,2? kiểm ở Cách 3)")

[3, 4, 2, 1, 0] (hợp lệ) [3, 2, 4, 0, 1] (chứa 3,1,2? kiểm ở Cách 3)


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1: `pushed = [1,2,3,4,5]`, `popped = [4,5,3,2,1]`.

| thao tác | ngăn xếp | cần pop (`popped[j]`) |
|---|---|---|
| push 1,2,3,4 | `[1,2,3,4]` | 4 → đỉnh = 4: pop |
| sau pop | `[1,2,3]` | 5 → đỉnh 3 ≠ 5: đẩy 5 |
| push 5 | `[1,2,3,5]` | 5 → pop |
| `[1,2,3]` | | 3 → pop, rồi 2, rồi 1 |

Ví dụ 2: `popped = [4,3,5,1,2]`: sau `pop 4`, `pop 3`, cần `5`: đẩy `5` rồi `pop 5`, ngăn xếp `[1,2]` cần `1` nhưng đỉnh là `2` và `pushed` đã hết ⇒ `false`.

In [4]:
def simulate_verbose(pushed, popped):
    stack, j = [], 0
    for x in pushed:
        stack.append(x)
        print(f"push {x}: {stack}")
        while stack and j < len(popped) and stack[-1] == popped[j]:
            print(f"   đỉnh {stack[-1]} == popped[{j}] -> pop")
            stack.pop()
            j += 1
    print("kết thúc: ngăn xếp", stack, "| đã pop", j, "/", len(popped))
    return not stack


simulate_verbose([1, 2, 3, 4, 5], [4, 3, 5, 1, 2])

push 1: [1]
push 2: [1, 2]
push 3: [1, 2, 3]
push 4: [1, 2, 3, 4]
   đỉnh 4 == popped[0] -> pop
   đỉnh 3 == popped[1] -> pop
push 5: [1, 2, 5]
   đỉnh 5 == popped[2] -> pop
kết thúc: ngăn xếp [1, 2] | đã pop 3 / 5


False

## 5. Cách 1 — Mô phỏng tham lam

**Ý tưởng:** duyệt `pushed`; sau mỗi `push`, **trong khi** đỉnh bằng `popped[j]`: `pop`, `j += 1`. Cuối cùng trả `ngăn xếp rỗng` (tương đương `j == n`).

In [5]:
def validate_greedy(pushed, popped):
    stack, j = [], 0
    for x in pushed:
        stack.append(x)                                  # đẩy phần tử kế tiếp
        while stack and stack[-1] == popped[j]:          # trong khi đỉnh đúng là phần tử cần lấy: lấy ngay
            stack.pop()
            j += 1
    return not stack                                     # mọi phần tử đã được lấy ra


for (a, b), want in examples:
    assert validate_greedy(a, b) == want
print([validate_greedy(a, b) for (a, b), _ in examples])

[True, False]


In [6]:
# Code nộp LeetCode cho Cách 1
class SolutionGreedy:
    def validateStackSequences(self, pushed: List[int], popped: List[int]) -> bool:
        stack, j = [], 0
        for x in pushed:
            stack.append(x)
            while stack and stack[-1] == popped[j]:
                stack.pop()
                j += 1
        return not stack

**Độ phức tạp:** `O(n)` thời gian (mỗi phần tử push một lần, pop tối đa một lần), `O(n)` bộ nhớ.

(Chú ý: khi `stack` rỗng ta không truy cập `popped[j]` nhờ `and` ngắn mạch; khi `j = n` thì `stack` chắc chắn rỗng nên cũng không truy cập quá mảng.)

**Nút thắt:** cần ngăn xếp thật `O(n)`; có thể dùng chính mảng `pushed` làm ngăn xếp.

## 6. Ý tưởng thuật toán / bước đột phá

Tham lam: pop **ngay khi có thể** (mục 3.3). Hai biến thể tương đương: dùng mảng `pushed` làm ngăn xếp (con trỏ `top`) cho `O(1)` bộ nhớ thêm; hoặc kiểm tra mẫu `312`; hoặc xét trạng thái `(i, j)` (không giả định tham lam).

## 7. Các cách tiếp theo

Ngoài Cách 1 (mục 5), ta có thêm 3 cách: **Cách 2** dùng `pushed` làm ngăn xếp tại chỗ, **Cách 3** kiểm tra mẫu `312` bị cấm, **Cách 4** quy hoạch động trên trạng thái `(i, j)`.

### Cách 2 — Dùng `pushed` làm ngăn xếp (`O(1)` bộ nhớ thêm)

**Ý tưởng:** con trỏ `top` (kích thước ngăn xếp) trong chính mảng `pushed` (ghi đè các ô đã dùng). Với mỗi `x` của `pushed`: `pushed[top] = x; top += 1`; trong khi `top > 0 and pushed[top − 1] == popped[j]`: `top -= 1; j += 1`. Vì `top ≤` chỉ số hiện tại nên không ghi đè phần tử chưa đọc.

In [7]:
def validate_in_place(pushed, popped):
    pushed = list(pushed)                                # sao chép để không phá đầu vào của người gọi
    top, j = 0, 0                                        # top: số phần tử trong ngăn xếp (các ô pushed[0:top])
    for x in list(pushed):                               # duyệt bản sao: ta ghi đè pushed khi dùng làm ngăn xếp
        pushed[top] = x
        top += 1
        while top and pushed[top - 1] == popped[j]:
            top -= 1
            j += 1
    return top == 0


for (a, b), want in examples:
    assert validate_in_place(a, b) == want
print([validate_in_place(a, b) for (a, b), _ in examples])

[True, False]


In [8]:
# Code nộp LeetCode cho Cách 2 (dùng pushed làm ngăn xếp: không cấp phát danh sách mới)
class SolutionInPlace:
    def validateStackSequences(self, pushed: List[int], popped: List[int]) -> bool:
        top = j = 0
        for i in range(len(pushed)):
            pushed[top] = pushed[i]                      # top <= i nên không ghi đè phần tử chưa đọc
            top += 1
            while top and pushed[top - 1] == popped[j]:
                top -= 1
                j += 1
        return top == 0

**Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ thêm (ghi đè đầu vào).

### Cách 3 — Kiểm tra mẫu `312` bị cấm

**Ý tưởng:** đổi `popped` thành dãy vị trí `q` trong `pushed`. Hợp lệ ⇔ không có `a < b < c` với `q_b < q_c < q_a`. Kiểm tra `O(n²)`: với mỗi `b`, đặt `M = max(q[:b])`; nếu `M > q_b` thì nếu tồn tại `c > b` với `q_b < q_c < M` ⇒ có mẫu `312` ⇒ không hợp lệ.

In [9]:
def validate_pattern(pushed, popped):
    pos = {v: i for i, v in enumerate(pushed)}
    q = [pos[v] for v in popped]
    n = len(q)
    prefix_max = -1
    for b in range(n):
        if prefix_max > q[b]:                            # có phần tử "lớn" a đứng trước b (q_a > q_b)
            for c in range(b + 1, n):
                if q[b] < q[c] < prefix_max:             # và c sau b với q_b < q_c < q_a: mẫu 3-1-2
                    return False
        prefix_max = max(prefix_max, q[b])
    return True


for (a, b), want in examples:
    assert validate_pattern(a, b) == want
print([validate_pattern(a, b) for (a, b), _ in examples])

[True, False]


In [10]:
# Code nộp LeetCode cho Cách 3 (O(n^2))
class SolutionPattern:
    def validateStackSequences(self, pushed: List[int], popped: List[int]) -> bool:
        pos = {v: i for i, v in enumerate(pushed)}
        q = [pos[v] for v in popped]
        n, prefix_max = len(q), -1
        for b in range(n):
            if prefix_max > q[b]:
                for c in range(b + 1, n):
                    if q[b] < q[c] < prefix_max:
                        return False
            prefix_max = max(prefix_max, q[b])
        return True

**Độ phức tạp:** `O(n²)` thời gian (có thể `O(n log n)` bằng cấu trúc dữ liệu), `O(n)` bộ nhớ. Cho thấy **cấu trúc** của các dãy lấy ra hợp lệ.

### Cách 4 — Quy hoạch động trên trạng thái `(i, j)`

**Ý tưởng:** không giả định tham lam. Trạng thái `(i, j)`: đã đẩy `i` phần tử, đã lấy `j` phần tử ⇒ ngăn xếp xác định (`pushed[:i]` trừ `popped[:j]`). Từ `(i, j)`: **đẩy** nếu `i < n` ⇒ `(i + 1, j)`; **lấy** nếu ngăn xếp không rỗng và đỉnh `== popped[j]` ⇒ `(i, j + 1)`. Hợp lệ ⇔ tới được `(n, n)`. Đỉnh ngăn xếp tại `(i, j)` tính bằng cách giữ thêm ngăn xếp, hoặc suy ra từ dữ liệu — ta dùng tìm kiếm có nhớ trên `(i, j)` và truyền ngăn xếp dạng tuple.

In [11]:
def validate_dp(pushed, popped):
    n = len(pushed)

    @lru_cache(None)
    def go(i, j, stack):                                 # stack là tuple: nội dung ngăn xếp (xác định bởi (i, j))
        if j == n:
            return True
        if i < n and go(i + 1, j, stack + (pushed[i],)):                     # lựa chọn 1: đẩy
            return True
        if stack and stack[-1] == popped[j] and go(i, j + 1, stack[:-1]):    # lựa chọn 2: lấy
            return True
        return False

    return go(0, 0, ())


for (a, b), want in examples:
    assert validate_dp(a, b) == want
print([validate_dp(a, b) for (a, b), _ in examples])

[True, False]


In [12]:
# Code nộp LeetCode cho Cách 4 (O(n^2) trạng thái, mỗi trạng thái mang tuple: chậm hơn nhưng không giả định tham lam)
from functools import lru_cache


class SolutionDP:
    def validateStackSequences(self, pushed: List[int], popped: List[int]) -> bool:
        n = len(pushed)

        @lru_cache(None)
        def go(i, j, stack):
            if j == n:
                return True
            if i < n and go(i + 1, j, stack + (pushed[i],)):
                return True
            return bool(stack) and stack[-1] == popped[j] and go(i, j + 1, stack[:-1])

        return go(0, 0, ())

**Độ phức tạp:** số trạng thái `(i, j)` tối đa `O(n²)` (ngăn xếp xác định bởi `(i, j)`), mỗi trạng thái xử lý `O(n)` (sao chép tuple) ⇒ `O(n³)` trong trường hợp xấu; dùng cho `n` nhỏ (oracle thứ hai). Độ sâu đệ quy tới `2n`.

## 8. TRACE từng bước

**Trace Cách 1** với ví dụ 1 (`pushed = [1,2,3,4,5]`, `popped = [4,5,3,2,1]`):

| thao tác | ngăn xếp | `j` | ghi chú |
|---|---|---|---|
| push 1 | `[1]` | 0 | đỉnh 1 ≠ 4 |
| push 2 | `[1,2]` | 0 | |
| push 3 | `[1,2,3]` | 0 | |
| push 4 | `[1,2,3,4]` | 0 | đỉnh 4 == `popped[0]` ⇒ pop |
| pop 4 | `[1,2,3]` | 1 | đỉnh 3 ≠ 5 |
| push 5 | `[1,2,3,5]` | 1 | đỉnh 5 == `popped[1]` ⇒ pop |
| pop 5 | `[1,2,3]` | 2 | đỉnh 3 == 3 ⇒ pop |
| pop 3 | `[1,2]` | 3 | đỉnh 2 == 2 ⇒ pop |
| pop 2 | `[1]` | 4 | đỉnh 1 == 1 ⇒ pop |
| pop 1 | `[]` | 5 | rỗng ⇒ `true` |

Hàm trace in từng bước (đã định nghĩa `simulate_verbose` ở mục 4). **Trace Cách 3** với ví dụ 2: `q = [3,2,4,0,1]`; `b = 3` (`q_b = 0`): `prefix_max = 4 > 0`; tồn tại `c = 4` với `0 < q_c = 1 < 4` ⇒ mẫu `312` (các giá trị `4, 1, 2`... tức `popped` chứa `5` rồi `1` rồi `2`) ⇒ `false` ✓.

In [13]:
res = simulate_verbose([1, 2, 3, 4, 5], [4, 5, 3, 2, 1])
assert res is True
print()
q = pushed_positions([1, 2, 3, 4, 5], [4, 3, 5, 1, 2])
print("q =", q, "-> mẫu 312 tại (q_a, q_b, q_c) = (4, 0, 1):", validate_pattern([1, 2, 3, 4, 5], [4, 3, 5, 1, 2]) is False)

push 1: [1]
push 2: [1, 2]
push 3: [1, 2, 3]
push 4: [1, 2, 3, 4]
   đỉnh 4 == popped[0] -> pop
push 5: [1, 2, 3, 5]
   đỉnh 5 == popped[1] -> pop
   đỉnh 3 == popped[2] -> pop
   đỉnh 2 == popped[3] -> pop
   đỉnh 1 == popped[4] -> pop
kết thúc: ngăn xếp [] | đã pop 5 / 5

q = [3, 2, 4, 0, 1] -> mẫu 312 tại (q_a, q_b, q_c) = (4, 0, 1): True


## 9. Code nộp LeetCode

Chọn **Cách 1** (mô phỏng tham lam). Giữ nguyên `class Solution`, `validateStackSequences(self, pushed, popped)`:

In [14]:
class Solution:
    def validateStackSequences(self, pushed: List[int], popped: List[int]) -> bool:
        stack, j = [], 0                                   # j: chỉ số phần tử cần lấy ra tiếp theo
        for x in pushed:
            stack.append(x)                                # đẩy phần tử kế tiếp
            while stack and stack[-1] == popped[j]:        # đỉnh khớp: lấy ngay (là lựa chọn duy nhất khả thi)
                stack.pop()
                j += 1
        return not stack                                   # hợp lệ nếu mọi phần tử đã được lấy ra


for (a, b), want in examples:
    assert Solution().validateStackSequences(list(a), list(b)) == want
print([Solution().validateStackSequences(list(a), list(b)) for (a, b), _ in examples])

[True, False]


## 10. Vì sao đúng

**(a) Điều kiện cần của mô phỏng.** Mọi dãy thao tác hợp lệ đưa `pushed` vào theo thứ tự và lấy ra theo `popped`. Tại mỗi thời điểm ngăn xếp là một dãy con của `pushed` (theo thứ tự đẩy).

**(b) Pop ngay là bắt buộc (khi có thể).** Xét một dãy thao tác hợp lệ `S`. Giả sử ở một thời điểm đỉnh ngăn xếp bằng `popped[j]` (phần tử cần ra tiếp theo) nhưng `S` đẩy thêm `y` trước khi pop. Khi đó `popped[j]` nằm dưới `y` trong ngăn xếp nên chỉ ra được **sau** `y`; nhưng `y` còn thuộc phần chưa ra (`y ∈ popped[j+1:]`, vì các giá trị phân biệt và `y ≠ popped[j]`) nên `y` ra sau `popped[j]` — mâu thuẫn. Vậy mọi dãy hợp lệ **pop ngay** khi đỉnh bằng `popped[j]`. Do đó chiến lược "pop ngay khi có thể" đúng khi tồn tại nghiệm; nếu nó kẹt (đỉnh ≠ `popped[j]` và `pushed` hết) thì không có nghiệm.

**(c) Kết luận của Cách 1.** Mô phỏng sinh ra **duy nhất** một dãy thao tác "pop ngay"; nếu nó lấy ra hết thì hợp lệ (`true`), ngược lại không có dãy hợp lệ nào (`false`). Cuối cùng `not stack` ⇔ `j = n`.

**(d) Cách 2.** `top ≤ i` luôn đúng (ngăn xếp có tối đa `i + 1` phần tử sau khi đẩy phần tử thứ `i`), nên ghi `pushed[top]` không ghi đè phần tử chưa đọc.

**(e) Cách 3.** Đánh số `pushed` bằng `0..n−1`; `q_k` là vị trí trong `pushed` của phần tử ra thứ `k`. *Hợp lệ ⇒ tránh `312`:* giả sử `a < b < c` với `q_b < q_c < q_a`. Khi phần tử `q_a` được lấy ra (thời điểm `a`), `q_c` đã được đẩy (vì `q_c < q_a` nên đẩy trước) và chưa ra (vì `c > a`), nên nó nằm trong ngăn xếp; tương tự `q_b < q_c` đã được đẩy trước `q_c` và chưa ra (`b > a`), nên `q_b` cũng nằm trong ngăn xếp, **dưới** `q_c` (đẩy trước). Do LIFO, `q_c` phải ra trước `q_b`, mâu thuẫn với `b < c`. Chiều ngược lại (tránh `312` ⇒ hợp lệ) là kết quả kinh điển của Knuth; ta kiểm chứng bằng máy: số hoán vị tránh `312` bằng số Catalan và khớp mô phỏng trên mọi hoán vị `n ≤ 8`.

**(f) Cách 4.** Là tìm kiếm trên mọi dãy thao tác có thể (đẩy/lấy), **không** giả định tham lam; nên đồng thời là kiểm chứng độc lập cho (b).

Ta kiểm chứng bằng máy: mọi cách khớp oracle tìm-kiếm-vét-cạn trên **mọi hoán vị** của `0..n−1` với `n ≤ 7` (số hợp lệ đúng Catalan), và `n = 8` cho các cách nhanh (mục 12D).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Mô phỏng tham lam | `O(n)` | `O(n)` | **đáp án chính** |
| 2. `pushed` làm ngăn xếp | `O(n)` | `O(1)` thêm | ghi đè đầu vào |
| 3. Mẫu `312` bị cấm | `O(n²)` | `O(n)` | cho thấy cấu trúc |
| 4. DP / tìm kiếm `(i, j)` | `O(n³)` xấu | `O(n²)` | oracle, `n` nhỏ |

Đo thực tế (`n = 1000`; cách 4 chỉ `n = 40`):

In [15]:
def valid_instance(rng, n):
    """Sinh (pushed, popped) hợp lệ bằng một dãy thao tác push/pop ngẫu nhiên; giá trị là hoán vị ngẫu nhiên của 0..n-1."""
    vals = list(range(n))
    rng.shuffle(vals)
    pushed, popped, stack, i = vals[:], [], [], 0
    while len(popped) < n:
        if i < n and (not stack or rng.random() < 0.55):
            stack.append(vals[i])
            i += 1
        else:
            popped.append(stack.pop())
    return pushed, popped


def bench(fn, args, repeat=3):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(*[list(x) for x in args])
        best = min(best, time.perf_counter() - t0)
    return best


rng = random.Random(946)
inst = valid_instance(rng, 1000)
for name, fn in [("greedy", validate_greedy), ("in_place", validate_in_place), ("pattern", validate_pattern)]:
    print(f"n = 1000 {name:9s} {bench(fn, inst) * 1e3:8.3f} ms")
small = valid_instance(rng, 40)
print(f"dp n = 40: {bench(validate_dp, small, 1) * 1e3:8.2f} ms")

n = 1000 greedy       0.125 ms
n = 1000 in_place     0.151 ms
n = 1000 pattern     10.241 ms
dp n = 40:     0.49 ms


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn mọi hoán vị với oracle tìm kiếm (số hợp lệ = Catalan), (E) ngẫu nhiên có seed + tính chất, (F) giới hạn.

### A. Ví dụ của đề

In [16]:
def oracle(pushed, popped):
    """Tìm kiếm vét cạn mọi dãy thao tác: ở mỗi bước thử CẢ HAI lựa chọn (đẩy / lấy) — không giả định tham lam."""
    n = len(pushed)
    seen = set()

    def dfs(i, j, stack):
        if j == n:
            return True
        key = (i, j, stack)
        if key in seen:
            return False
        seen.add(key)
        if i < n and dfs(i + 1, j, stack + (pushed[i],)):
            return True
        if stack and stack[-1] == popped[j] and dfs(i, j + 1, stack[:-1]):
            return True
        return False

    return dfs(0, 0, ())


IMPLS = {
    "greedy": validate_greedy,
    "in_place": validate_in_place,
    "pattern": validate_pattern,
    "dp": validate_dp,
    "Solution": lambda a, b: Solution().validateStackSequences(list(a), list(b)),
    "SolutionGreedy": lambda a, b: SolutionGreedy().validateStackSequences(list(a), list(b)),
    "SolutionInPlace": lambda a, b: SolutionInPlace().validateStackSequences(list(a), list(b)),
    "SolutionPattern": lambda a, b: SolutionPattern().validateStackSequences(list(a), list(b)),
    "SolutionDP": lambda a, b: SolutionDP().validateStackSequences(list(a), list(b)),
}


def check(pushed, popped, want=None, names=None):
    if want is None:
        want = oracle(pushed, popped)
    for name in (names or IMPLS):
        got = IMPLS[name](list(pushed), list(popped))
        assert got == want, (name, pushed, popped, got, want)


for (a, b), want in examples:
    assert oracle(a, b) == want
    check(a, b, want)
print("ví dụ đề: oracle tìm kiếm vét cạn và", len(IMPLS), "cài đặt đúng")

ví dụ đề: oracle tìm kiếm vét cạn và 9 cài đặt đúng


### B. Edge case

- Một phần tử.
- `popped == pushed` (đẩy rồi lấy ngay từng phần tử).
- `popped` là `pushed` đảo ngược (đẩy hết rồi lấy hết).
- Hai phần tử: cả hai thứ tự hợp lệ.
- Mẫu cấm tối thiểu: `pushed = [1,2,3]`, `popped = [3,1,2]` ⇒ `false`.
- Giá trị `0` và `1000`.
- Dãy lấy ra xen kẽ lên xuống hợp lệ.

In [17]:
cases = [
    ([1], [1], True), ([1, 2], [1, 2], True), ([1, 2], [2, 1], True),
    ([1, 2, 3], [1, 2, 3], True), ([1, 2, 3], [3, 2, 1], True), ([1, 2, 3], [3, 1, 2], False), ([1, 2, 3], [2, 3, 1], True),
    ([1, 2, 3], [1, 3, 2], True), ([1, 2, 3], [2, 1, 3], True), ([0, 1000], [1000, 0], True),
    ([2, 1, 0], [1, 2, 0], True), ([4, 0, 3, 1, 2], [1, 3, 0, 2, 4], None),
]
for a, b, want in cases:
    check(a, b, want)
print(len(cases), "edge case: oracle và mọi cài đặt đúng")

12 edge case: oracle và mọi cài đặt đúng


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [18]:
def bug_if_not_while(pushed, popped):                  # pop tối đa MỘT lần sau mỗi push
    stack, j = [], 0
    for x in pushed:
        stack.append(x)
        if stack and stack[-1] == popped[j]:
            stack.pop()
            j += 1
    return not stack


def bug_no_final_check(pushed, popped):                # không kiểm tra ngăn xếp rỗng cuối cùng
    stack, j = [], 0
    for x in pushed:
        stack.append(x)
        while stack and stack[-1] == popped[j]:
            stack.pop()
            j += 1
    return True


def bug_push_all_then_pop(pushed, popped):             # đẩy hết rồi mới lấy: chỉ đúng khi popped là đảo ngược
    return list(reversed(pushed)) == list(popped)


def bug_pop_before_push(pushed, popped):               # kiểm tra pop TRƯỚC khi push (lệch một nhịp)
    stack, j = [], 0
    for x in pushed:
        while stack and stack[-1] == popped[j]:
            stack.pop()
            j += 1
        stack.append(x)
    return not stack


def bug_compare_sorted(pushed, popped):                # so sánh tập hợp (luôn True vì popped là hoán vị)
    return sorted(pushed) == sorted(popped)


def bug_stack_empty_index(pushed, popped):             # không chặn j: dùng try để che lỗi và trả False
    stack, j = [], 0
    for x in pushed:
        stack.append(x)
        while stack and j < len(popped) and stack[-1] == popped[j]:
            stack.pop()
            j += 1
    return j == len(popped) - 1                        # SAI: lệch một (phải là == len(popped))


BUGS = [bug_if_not_while, bug_no_final_check, bug_push_all_then_pop, bug_pop_before_push, bug_compare_sorted, bug_stack_empty_index]
bad_cases = {}
for bug in BUGS:
    found = None
    for n in range(1, 6):
        for perm in itertools.permutations(range(n)):
            pushed = list(range(n))
            if bug(pushed, list(perm)) != oracle(pushed, list(perm)):
                found = (pushed, list(perm))
                break
        if found:
            break
    assert found is not None, bug.__name__
    bad_cases[bug.__name__] = found
    print(f"{bug.__name__:24s} sai trên pushed = {found[0]}, popped = {found[1]}: ra {bug(*found)}, đúng {oracle(*found)}")

bug_if_not_while         sai trên pushed = [0, 1], popped = [1, 0]: ra False, đúng True
bug_no_final_check       sai trên pushed = [0, 1, 2], popped = [2, 0, 1]: ra True, đúng False
bug_push_all_then_pop    sai trên pushed = [0, 1], popped = [0, 1]: ra False, đúng True
bug_pop_before_push      sai trên pushed = [0], popped = [0]: ra False, đúng True
bug_compare_sorted       sai trên pushed = [0, 1, 2], popped = [2, 0, 1]: ra True, đúng False
bug_stack_empty_index    sai trên pushed = [0], popped = [0]: ra False, đúng True


### D. Vét cạn mọi hoán vị

Với `pushed = [0, 1, …, n−1]` và **mọi** hoán vị `popped` của `n` phần tử, `n = 1..7` (`1 + 2 + 6 + 24 + 120 + 720 + 5 040 = 5 913` hoán vị): mọi cài đặt khớp oracle tìm kiếm, và **số hoán vị hợp lệ** đúng là số Catalan `1, 2, 5, 14, 42, 132, 429`. Với `n = 8` (`40 320` hoán vị, số hợp lệ `1 430`) chỉ chạy các cách nhanh (tham lam, tại chỗ, mẫu `312`).

In [19]:
t0 = time.perf_counter()
catalan = {1: 1, 2: 2, 3: 5, 4: 14, 5: 42, 6: 132, 7: 429, 8: 1430}
cnt = 0
fast = ("greedy", "in_place", "pattern", "Solution")
for n in range(1, 8):
    pushed = list(range(n))
    valid = 0
    for perm in itertools.permutations(range(n)):
        popped = list(perm)
        want = oracle(pushed, popped)
        for name in fast + (("dp",) if n <= 6 else ()):
            assert IMPLS[name](pushed, popped) == want, (name, popped)
        valid += want
        cnt += 1
    assert valid == catalan[n], (n, valid)
valid8 = 0
pushed8 = list(range(8))
for perm in itertools.permutations(range(8)):
    popped = list(perm)
    r = validate_greedy(pushed8, popped)
    assert r == validate_in_place(pushed8, popped) == validate_pattern(pushed8, popped)
    valid8 += r
    cnt += 1
assert valid8 == catalan[8]
print(cnt, "hoán vị: các cách khớp oracle; số hợp lệ đúng Catalan 1,2,5,14,42,132,429,1430;", round(time.perf_counter() - t0, 1), "s")

46233 hoán vị: các cách khớp oracle; số hợp lệ đúng Catalan 1,2,5,14,42,132,429,1430; 0.3 s


### E. Ngẫu nhiên có seed và tính chất

Dữ liệu hợp lệ sinh từ dãy thao tác ngẫu nhiên (`true` chắc chắn) và dữ liệu bị phá (hoán đổi hai phần tử của `popped`, nhãn bằng oracle/pattern). Tính chất: (1) đổi nhãn giá trị (hoán vị giá trị) không đổi kết quả; (2) ghép thêm phần tử mới vào cuối cả hai dãy (`pushed + [v]`, `popped + [v]`) không đổi kết quả; (3) `true` ⇒ sinh lại được một dãy thao tác đúng (kiểm tra bằng mô phỏng ghi lại thao tác).

In [20]:
def operations(pushed, popped):
    """Trả danh sách thao tác ('push', x)/('pop', x) của mô phỏng tham lam (hoặc None nếu không hợp lệ)."""
    stack, j, ops = [], 0, []
    for x in pushed:
        stack.append(x)
        ops.append(("push", x))
        while stack and stack[-1] == popped[j]:
            ops.append(("pop", stack.pop()))
            j += 1
    return ops if not stack else None


def replay(ops, pushed, popped):
    stack, out, pushed_order = [], [], []
    for op, x in ops:
        if op == "push":
            stack.append(x)
            pushed_order.append(x)
        else:
            assert stack and stack[-1] == x
            out.append(stack.pop())
    return not stack and pushed_order == pushed and out == popped


rng = random.Random(20240946)
for _ in range(1500):
    n = rng.randint(1, 8)
    pushed, popped = valid_instance(rng, n)
    check(pushed, popped, True, names=("greedy", "in_place", "pattern", "dp", "Solution"))
    ops = operations(pushed, popped)
    assert ops is not None and replay(ops, pushed, popped)                                      # (3)
    bad = popped[:]
    if n >= 2:
        i, j = rng.sample(range(n), 2)
        bad[i], bad[j] = bad[j], bad[i]
    want = oracle(pushed, bad)
    check(pushed, bad, want)
    relabel = list(range(100, 100 + n))
    rng.shuffle(relabel)
    mp = {v: relabel[v] for v in range(n)}
    assert Solution().validateStackSequences([mp[v] for v in pushed], [mp[v] for v in bad]) == want        # (1)
    assert Solution().validateStackSequences(pushed + [n], bad + [n]) == want                              # (2)
print("1500 dữ liệu: 9 cài đặt khớp oracle (kể cả dữ liệu bị phá); tính chất đổi nhãn, nối phần tử cuối, dựng lại thao tác đúng")

1500 dữ liệu: 9 cài đặt khớp oracle (kể cả dữ liệu bị phá); tính chất đổi nhãn, nối phần tử cuối, dựng lại thao tác đúng


### F. Giới hạn: `n = 1000`

Dữ liệu hợp lệ lớn (đẩy hết rồi lấy hết, đẩy-lấy xen kẽ, ngẫu nhiên), dữ liệu bị phá, và các cách nhanh (tham lam, tại chỗ, mẫu `312`) khớp nhau.

In [21]:
rng = random.Random(500)
n = 1000
cases = {}
vals = list(range(n))
rng.shuffle(vals)
cases["đẩy hết rồi lấy hết"] = (vals, vals[::-1], True)
cases["đẩy-lấy xen kẽ"] = (vals, vals[:], True)
rp, rq = valid_instance(rng, n)
cases["ngẫu nhiên hợp lệ"] = (rp, rq, True)
broken = rq[:]
broken[0], broken[-1] = broken[-1], broken[0]
cases["bị phá (đổi hai đầu)"] = (rp, broken, None)
cases["mẫu 312 cuối"] = (list(range(n)), list(range(n - 3)) + [n - 1, n - 3, n - 2], False)
fast = ["greedy", "in_place", "pattern", "Solution", "SolutionGreedy", "SolutionInPlace", "SolutionPattern"]
for name, (a, b, want) in cases.items():
    res = {k: IMPLS[k](a, b) for k in fast}
    assert len(set(res.values())) == 1, (name, res)
    if want is not None:
        assert res["greedy"] == want, name
    print(f"{name:24s}: {res['greedy']}")
print("n = 1000: 7 cài đặt nhanh khớp nhau và khớp kỳ vọng")

đẩy hết rồi lấy hết     : True
đẩy-lấy xen kẽ          : True


ngẫu nhiên hợp lệ       : True
bị phá (đổi hai đầu)    : False
mẫu 312 cuối            : False
n = 1000: 7 cài đặt nhanh khớp nhau và khớp kỳ vọng


## 13. Các lỗi người mới hay gặp

### Lỗi 1: `if` thay vì `while` khi pop

In [22]:
a, b = bad_cases["bug_if_not_while"]
print(a, b, "-> if:", bug_if_not_while(a, b), "| đúng:", Solution().validateStackSequences(list(a), list(b)))
assert bug_if_not_while([1, 2, 3], [3, 2, 1]) is False and Solution().validateStackSequences([1, 2, 3], [3, 2, 1]) is True

[0, 1] [1, 0] -> if: False | đúng: True


**Vì sao sai:** sau một lần pop, đỉnh mới có thể lại khớp phần tử cần lấy tiếp theo. **Đúng:** vòng `while`.

### Lỗi 2: quên kiểm tra ngăn xếp rỗng ở cuối

In [23]:
a, b = bad_cases["bug_no_final_check"]
print(a, b, "-> luôn True:", bug_no_final_check(a, b), "| đúng:", Solution().validateStackSequences(list(a), list(b)))
assert bug_no_final_check([1, 2, 3], [3, 1, 2]) is True and Solution().validateStackSequences([1, 2, 3], [3, 1, 2]) is False

[0, 1, 2] [2, 0, 1] -> luôn True: True | đúng: False


**Vì sao sai:** tham lam có thể kẹt (đỉnh không khớp và `pushed` hết) mà vẫn trả `True`. **Đúng:** `return not stack`.

### Lỗi 3: đẩy hết rồi mới lấy

In [24]:
a, b = bad_cases["bug_push_all_then_pop"]
print(a, b, "->", bug_push_all_then_pop(a, b), "| đúng:", Solution().validateStackSequences(list(a), list(b)))
assert bug_push_all_then_pop([1, 2, 3], [1, 2, 3]) is False and Solution().validateStackSequences([1, 2, 3], [1, 2, 3]) is True

[0, 1] [0, 1] -> False | đúng: True


**Vì sao sai:** đề cho phép **xen kẽ** `push` và `pop`. **Đúng:** mô phỏng xen kẽ.

### Lỗi 4: kiểm tra pop **trước** khi push (lệch một nhịp)

In [25]:
a, b = bad_cases["bug_pop_before_push"]
print(a, b, "->", bug_pop_before_push(a, b), "| đúng:", Solution().validateStackSequences(list(a), list(b)))

[0] [0] -> False | đúng: True


**Vì sao sai:** phần tử vừa `push` có thể ngay lập tức cần `pop`. **Đúng:** `push` rồi mới `while ... pop`.

### Lỗi 5: chỉ so sánh tập hợp

In [26]:
print("[1,2,3] vs [3,1,2]:", bug_compare_sorted([1, 2, 3], [3, 1, 2]), "| đúng:", Solution().validateStackSequences([1, 2, 3], [3, 1, 2]))
assert bug_compare_sorted([1, 2, 3], [3, 1, 2]) is True

[1,2,3] vs [3,1,2]: True | đúng: False


**Vì sao sai:** `popped` luôn là hoán vị của `pushed` nên điều kiện đó luôn đúng; **thứ tự** mới là điều cần kiểm tra.

### Lỗi 6 (bonus): lệch một ở điều kiện cuối

In [27]:
a, b = bad_cases["bug_stack_empty_index"]
print(a, b, "->", bug_stack_empty_index(a, b), "| đúng:", Solution().validateStackSequences(list(a), list(b)))

[0] [0] -> False | đúng: True


**Vì sao sai:** hợp lệ ⇔ `j == len(popped)` (đã lấy **hết**), không phải `len − 1`.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "dãy vào/ra của ngăn xếp (hoặc hàng đợi) có hợp lệ không" → **mô phỏng tham lam**: thao tác bắt buộc làm ngay (pop khi đỉnh khớp). Giá trị phân biệt là điều kiện để tham lam đúng.

**Mẹo:**

1. Hai con trỏ: một cho dãy vào, một cho dãy ra.
2. `while` để pop liên tiếp.
3. Kết quả = ngăn xếp rỗng cuối cùng.
4. Khi giá trị **trùng** nhau, tham lam có thể sai: phải thử cả hai lựa chọn.
5. Số dãy ra hợp lệ của `n` phần tử là Catalan `C_n` (tránh mẫu `312`).
6. Oracle: tìm kiếm cả hai lựa chọn (đẩy/lấy).

**Họ bài cùng pattern:** Valid Parentheses (20), Min Stack (155), Remove All Adjacent Duplicates (1047), Daily Temperatures (739), Next Greater Element (496), Online Stock Span (901), Backspace String Compare (844).

## 15. Biến thể

1. **Giá trị có thể trùng nhau:** tham lam không còn đủ; cần tìm kiếm/DP.
2. **Đếm số dãy `popped` hợp lệ** (Catalan).
3. **Hàng đợi** thay cho ngăn xếp.
4. **Khôi phục dãy thao tác** (đã có `operations`).
5. **Số lần pop tối đa liên tiếp / độ sâu ngăn xếp lớn nhất.**

Cài biến thể 1 (có giá trị trùng, tìm kiếm) và 2 (đếm Catalan), 5 (độ sâu lớn nhất):

In [28]:
def validate_with_duplicates(pushed, popped):
    """Biến thể 1: giá trị có thể trùng; tìm kiếm vét cạn có nhớ trên (i, j, nội dung ngăn xếp)."""
    n = len(pushed)

    @lru_cache(None)
    def go(i, j, stack):
        if j == n:
            return True
        if i < n and go(i + 1, j, stack + (pushed[i],)):
            return True
        return bool(stack) and stack[-1] == popped[j] and go(i, j + 1, stack[:-1])

    return go(0, 0, ())


def greedy_may_fail_with_duplicates():
    """Tìm ví dụ có giá trị trùng mà tham lam (pop ngay) sai nhưng tìm kiếm đúng."""
    for n in range(1, 6):
        for pushed in itertools.product(range(3), repeat=n):
            for popped in set(itertools.permutations(pushed)):
                if validate_greedy(list(pushed), list(popped)) != validate_with_duplicates(list(pushed), list(popped)):
                    return list(pushed), list(popped)


def count_valid_orders(n):
    """Biến thể 2: số hoán vị popped hợp lệ cho pushed = 0..n-1 = Catalan(n), bằng DP (i, j)->đếm số dãy thao tác phân biệt."""
    @lru_cache(None)
    def ways(pushed_left, stack_size):                   # số cách hoàn tất từ trạng thái: còn pushed_left phần tử chưa đẩy, ngăn xếp có stack_size
        if pushed_left == 0 and stack_size == 0:
            return 1
        r = 0
        if pushed_left:
            r += ways(pushed_left - 1, stack_size + 1)
        if stack_size:
            r += ways(pushed_left, stack_size - 1)
        return r

    return ways(n, 0)


def max_stack_depth(pushed, popped):
    stack, j, best = [], 0, 0
    for x in pushed:
        stack.append(x)
        best = max(best, len(stack))
        while stack and stack[-1] == popped[j]:
            stack.pop()
            j += 1
    return best if not stack else None


ex = greedy_may_fail_with_duplicates()
print("có giá trị trùng: tham lam sai trên", ex, "| tìm kiếm đúng:", validate_with_duplicates(*ex), "| tham lam:", validate_greedy(*ex))
assert [count_valid_orders(n) for n in range(1, 9)] == [1, 2, 5, 14, 42, 132, 429, 1430]
assert max_stack_depth([1, 2, 3, 4, 5], [4, 5, 3, 2, 1]) == 4 and max_stack_depth([1, 2, 3], [1, 2, 3]) == 1
print("đếm Catalan (DP), độ sâu lớn nhất: đúng")

có giá trị trùng: tham lam sai trên ([0, 1, 0, 2], [0, 2, 1, 0]) | tìm kiếm đúng: True | tham lam: False
đếm Catalan (DP), độ sâu lớn nhất: đúng


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng tham lam | push rồi pop khi đỉnh khớp | `O(n)` | `O(n)` | **đáp án chính** |
| 2. `pushed` làm ngăn xếp | ghi đè đầu vào | `O(n)` | `O(1)` thêm | tiết kiệm bộ nhớ |
| 3. Mẫu `312` bị cấm | `q` tránh `3-1-2` | `O(n²)` | `O(n)` | hiểu cấu trúc |
| 4. DP / tìm kiếm `(i, j)` | thử cả đẩy lẫn lấy | `O(n³)` xấu | `O(n²)` | oracle, giá trị trùng |

Cả bốn khớp oracle tìm kiếm trên 5 913 hoán vị, số hợp lệ đúng Catalan, và khớp nhau ở `n = 1000` (mục 12).

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [29]:
print(Solution().validateStackSequences([1, 2, 3], [2, 1, 3]), Solution().validateStackSequences([1, 2, 3], [3, 1, 2]), Solution().validateStackSequences([1, 0], [1, 0]), Solution().validateStackSequences([2, 1, 0], [1, 2, 0]))

True False True True


### Bài 2 (tính tay)

`pushed = [1,2,3,4]`, `popped = [2,1,4,3]`. (a) Ghi dãy thao tác. (b) Độ sâu ngăn xếp lớn nhất? (c) `popped = [3,1,2,4]` có hợp lệ không? Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện mô phỏng:

In [30]:
def simulate_todo(pushed, popped):
    stack, j = [], 0
    for x in pushed:
        pass            # TODO 1: push x
        pass            # TODO 2: while đỉnh == popped[j]: pop, j += 1
    return False        # TODO 3: trả True nếu ngăn xếp rỗng


# Khi làm xong, bỏ comment để kiểm tra:
# assert [simulate_todo(a, b) for (a, b), _ in examples] == [True, False]

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **`pushed` làm ngăn xếp** (`top`):

In [31]:
class MySolution:
    def validateStackSequences(self, pushed, popped):
        pass            # TODO


# Khi làm xong:
# assert all(MySolution().validateStackSequences(list(a), list(b)) == w for (a, b), w in examples)

### Bài 5 (mini challenge): Backspace String Compare (844)

Cho hai chuỗi `s`, `t` (chữ thường và `'#'` là phím xoá lùi), kiểm tra sau khi gõ chúng có bằng nhau không. Viết `backspace_compare(s, t)` bằng ngăn xếp và kiểm tra bằng mô phỏng chuỗi (`str`).

In [32]:
def backspace_compare(s, t):
    pass            # TODO


# Khi làm xong:
# assert backspace_compare("ab#c", "ad#c") and not backspace_compare("a#c", "b")

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: mô phỏng tham lam.
- Mức 2: `[1,2,3]` & `[2,1,3]`: push 1,2 → pop 2, pop 1, push 3 → pop 3 ⇒ `True`; `[3,1,2]` ⇒ `False`.
- Mức 3: kết quả `True False True True`: `[1,0]`&`[1,0]`: push 1 pop 1 push 0 pop 0; `[2,1,0]`&`[1,2,0]`: push 2,1 → pop 1 → pop 2 → push 0 → pop 0.

**Bài 2**

- Mức 1: push 1, push 2, pop 2, pop 1, push 3, push 4, pop 4, pop 3.
- Mức 2: độ sâu lớn nhất `2`.
- Mức 3: `[3,1,2,4]`: sau pop 3 ngăn xếp `[1,2]`, cần `1` nhưng đỉnh `2` ⇒ không hợp lệ (mẫu 312).

**Bài 3**

- Mức 1: `stack.append(x)`.
- Mức 2: `while stack and stack[-1] == popped[j]: stack.pop(); j += 1`.
- Mức 3: `return not stack`.

**Bài 4**

- Mức 1: `top = j = 0`.
- Mức 2: `pushed[top] = pushed[i]; top += 1`.
- Mức 3: `while top and pushed[top-1] == popped[j]: top -= 1; j += 1`; trả `top == 0`.

**Bài 5**

- Mức 1: xây từng chuỗi bằng ngăn xếp: `#` pop nếu có.
- Mức 2: so sánh hai ngăn xếp.
- Mức 3: kiểm tra bằng mô phỏng chuỗi `str`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [33]:
vals = (Solution().validateStackSequences([1, 2, 3], [2, 1, 3]), Solution().validateStackSequences([1, 2, 3], [3, 1, 2]), Solution().validateStackSequences([1, 0], [1, 0]), Solution().validateStackSequences([2, 1, 0], [1, 2, 0]))
assert vals == (True, False, True, True) == (oracle([1, 2, 3], [2, 1, 3]), oracle([1, 2, 3], [3, 1, 2]), oracle([1, 0], [1, 0]), oracle([2, 1, 0], [1, 2, 0]))
print("Bài 1 OK:", vals)

Bài 1 OK: (True, False, True, True)


### Bài 2

(a) `push 1, push 2, pop 2, pop 1, push 3, push 4, pop 4, pop 3`. (b) độ sâu lớn nhất `2`. (c) `[3,1,2,4]` **không** hợp lệ.

In [34]:
ops = operations([1, 2, 3, 4], [2, 1, 4, 3])
assert ops == [("push", 1), ("push", 2), ("pop", 2), ("pop", 1), ("push", 3), ("push", 4), ("pop", 4), ("pop", 3)]
assert max_stack_depth([1, 2, 3, 4], [2, 1, 4, 3]) == 2
assert Solution().validateStackSequences([1, 2, 3, 4], [3, 1, 2, 4]) is False and oracle([1, 2, 3, 4], [3, 1, 2, 4]) is False
print("Bài 2 OK")

Bài 2 OK


### Bài 3

In [35]:
def simulate_answer(pushed, popped):
    stack, j = [], 0
    for x in pushed:
        stack.append(x)
        while stack and stack[-1] == popped[j]:
            stack.pop()
            j += 1
    return not stack


assert [simulate_answer(a, b) for (a, b), _ in examples] == [True, False]
for n in range(1, 6):
    for perm in itertools.permutations(range(n)):
        assert simulate_answer(list(range(n)), list(perm)) == oracle(list(range(n)), list(perm))
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [36]:
class MySolutionAnswer:
    def validateStackSequences(self, pushed, popped):
        top = j = 0
        for i in range(len(pushed)):
            pushed[top] = pushed[i]
            top += 1
            while top and pushed[top - 1] == popped[j]:
                top -= 1
                j += 1
        return top == 0


assert all(MySolutionAnswer().validateStackSequences(list(a), list(b)) == w for (a, b), w in examples)
for n in range(1, 6):
    for perm in itertools.permutations(range(n)):
        assert MySolutionAnswer().validateStackSequences(list(range(n)), list(perm)) == oracle(list(range(n)), list(perm))
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [37]:
def backspace_compare_answer(s, t):
    def build(x):
        st = []
        for ch in x:
            if ch == "#":
                if st:
                    st.pop()
            else:
                st.append(ch)
        return st

    return build(s) == build(t)


def backspace_oracle(s, t):
    def typed(x):
        out = ""
        for ch in x:
            out = out[:-1] if ch == "#" else out + ch
        return out

    return typed(s) == typed(t)


assert backspace_compare_answer("ab#c", "ad#c") and not backspace_compare_answer("a#c", "b")
rng = random.Random(5)
for _ in range(500):
    s = "".join(rng.choice("ab#") for _ in range(rng.randint(0, 8)))
    t = "".join(rng.choice("ab#") for _ in range(rng.randint(0, 8)))
    assert backspace_compare_answer(s, t) == backspace_oracle(s, t)
print("Bài 5 OK: khớp mô phỏng chuỗi")

Bài 5 OK: khớp mô phỏng chuỗi


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (dãy vào/ra của ngăn xếp, giá trị phân biệt → mô phỏng tham lam):**

```python
class Solution:
    def validateStackSequences(self, pushed: List[int], popped: List[int]) -> bool:
        stack, j = [], 0
        for x in pushed:
            stack.append(x)
            while stack and stack[-1] == popped[j]:
                stack.pop()
                j += 1
        return not stack
```

**Checklist trước khi submit:**

- [ ] Sau mỗi `push`, dùng **`while`** (không `if`) để pop liên tiếp?
- [ ] Điều kiện `stack and stack[-1] == popped[j]` (kiểm tra rỗng trước)?
- [ ] Trả `not stack` (hoặc `j == len(popped)`), không luôn `True`?
- [ ] Đã dùng `push` **rồi** pop (không pop trước)?
- [ ] Giá trị phân biệt (nếu trùng phải dùng tìm kiếm/DP)?
- [ ] Đã thử 2 ví dụ, một phần tử, `popped = pushed`, `popped` đảo ngược, mẫu `312`?
- [ ] Đúng tên `class Solution`, `validateStackSequences(self, pushed, popped)`, không in debug.